In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "06-gateway/identity-security/agentic-identity-gcp-lab/notebooks/practice")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


    # 03 · Auth Manager: brokering credentials per call — practice

    **Primer sections:** §4.3, §5. Register providers, bind IAM, drive the consent flow by hand, read
    the dual-identity access log, then do the same through the ADK loop.

> **How to use this practice notebook.** Every `____` is a blank you must fill (a name, an
> argument, an expression); a `raise NotImplementedError("fill me")` means "write the body".
> Each exercise ends with `assert` checks — run the cell, and if it is silent you got it right.
> The completed version lives in `notebooks/solutions/`.

In [ ]:
import logging

from agentsec.logging_utils import quiet_logs

quiet_logs(logging.ERROR)

from urllib.parse import parse_qs, urlsplit

import jwt  # display only

from agentsec.agents import LocalStack, Step, reset_demo_state
from agentsec.config import Settings
from agentsec.identity import (
    AgentIdentity,
    LocalAuthManager,
    PermissionDenied,
    ProviderKind,
    TokenIssuer,
)
from agentsec.runtime import resume_after_auth, run_turn, seed_session
from agentsec.secrets import SecretValue

reset_demo_state()

def peek(token: str) -> dict:
    return jwt.decode(token, options={"verify_signature": False})

def short(spiffe: str) -> str:
    return spiffe.rsplit("/", 1)[-1]

ORG, PROJECT = "123456789012", "987654321098"
agent = AgentIdentity.for_agent_engine(project_number=PROJECT, location="us-central1", engine_id="support-agent", org_id=ORG)
other_agent = AgentIdentity.for_agent_engine(project_number="111111111111", location="us-central1", engine_id="marketing-agent", org_id=ORG)
issuer = TokenIssuer()
am = LocalAuthManager(issuer, project="demo-project", location="global")

## Exercise 1 — register a 3LO provider and an API-key provider

Create `crm-3lo` (audience `https://crm.acme.example`, allowed scope `crm.read`, the IdP URLs
below, client id `acme-support-agent`) and `weather-key` with the key `k-123-weather` stored as a
`SecretValue`.

In [ ]:
crm = am.create_provider(
    "crm-3lo", ProviderKind.____, audience="https://crm.acme.example",
    allowed_scopes=("crm.read",),
    authorization_url="https://idp.acme.example/o/oauth2/auth", token_url="https://idp.acme.example/o/oauth2/token",
    client_id="acme-support-agent",
)
weather = am.create_provider(
    "weather-key", ProviderKind.API_KEY, audience="https://weather.example",
    api_key=____("k-123-weather", name="weather-api-key"),
)
assert crm.name == "projects/demo-project/locations/global/authProviders/crm-3lo"
assert weather.kind is ProviderKind.API_KEY and "k-123-weather" not in repr(weather.api_key)
print(am.list_providers())

## Exercise 2 — IAM on the provider

Grant the support agent `roles/agentidentity.user` on `crm-3lo` (single principal) and every agent
in the project on `weather-key` (a `principalSet`). `marketing-agent` (other project) must be denied
on both; a disallowed scope must be denied too.

In [ ]:
am.add_iam_policy_binding(crm.name, agent.____)
am.add_iam_policy_binding(weather.name, f"principalSet://agents.global.org-{ORG}.system.id.goog/attribute.____/aiplatform/projects/{PROJECT}")

def denied(**kw) -> bool:
    try:
        am.retrieve_credentials(**kw)
        return False
    except ____:
        return True

assert denied(auth_provider=crm.name, user_id="u-ana", caller=other_agent, scopes=["crm.read"])
assert denied(auth_provider=weather.name, user_id=None, caller=other_agent)
assert denied(auth_provider=crm.name, user_id="u-ana", caller=agent, scopes=["crm.write"])
assert am.retrieve_credentials(auth_provider=weather.name, user_id=None, caller=agent).header == "X-API-Key"
print("IAM on the provider enforced; last log entries:", [e.outcome for e in am.access_log[-4:]])

## Exercise 3 — drive the consent flow by hand

Retrieve for `u-ana` (expect `uri_consent_required`), finalise with the nonce, retrieve again and
verify the token is delegated: `sub` is the user, `act.sub` is the agent, audience is the CRM.

In [ ]:
r1 = am.retrieve_credentials(auth_provider=crm.name, user_id="u-ana", caller=agent, scopes=["crm.read"])
assert r1.kind == "uri_consent_required" and r1.consent_nonce
params = parse_qs(urlsplit(r1.authorization_uri).query)
assert params["state"][0] == r1.consent_nonce
am.finalize(auth_provider=crm.name, user_id="u-ana", consent_nonce=____, user_id_validation_state="VALIDATED")
r2 = am.retrieve_credentials(auth_provider=crm.name, user_id="u-ana", caller=agent, scopes=["crm.read"])
assert r2.is_success and r2.header == "Authorization: Bearer"

claims = issuer.verify(r2.token, audience=____)
assert claims.subject == "u-ana" and claims.actor == agent.spiffe_id and claims.scopes == {"crm.read"}
assert claims.raw["authority"] == "delegated"
print("delegated CRM token for", claims.subject, "acting via", short(claims.actor))

## Exercise 4 — read the access log as an auditor

Write `attributable(log)` that returns the list of `(agent short name, user, outcome)` for every
successful 3LO retrieval. The successful CRM retrieval must show **both** identities.

In [ ]:
def attributable(log):
    raise NotImplementedError("fill me")  # [(short(e.agent), e.user, e.outcome) ...] for successful user-delegated retrievals

rows = attributable(am.access_log)
assert rows == [("support-agent", "u-ana", "success")]
assert "k-123-weather" not in repr(am.access_log)
for e in am.access_log:
    print(f"{short(e.agent):<16} {e.user or '-':<6} {e.provider.rsplit('/', 1)[-1]:<12} {e.outcome}")

## Exercise 5 — the ADK round trip

Run the reference agent so that it calls `crm_lookup`. The first turn must pause with an
`adk_request_credential`; finalise the consent with the stack's broker; resume; the tool result must
mention the user-delegated token.

In [ ]:
stack = LocalStack.create(Settings())
await seed_session(stack.runner, user_id="u-ana", session_id="s1",
                   user={"subject": "u-ana", "email": "ana@customer.example"}, scopes=["customers:read"])
stack.script(Step.call("crm_lookup", email="ana@customer.example"), Step.say("done"))
r = await run_turn(stack.runner, user_id="u-ana", session_id="s1", message="check the CRM")
assert len(r.pending_auth) == 1
pending = r.pending_auth[0]

stack.auth_manager.finalize(auth_provider=stack.crm_provider, user_id="u-ana", consent_nonce=____)
stack.script(Step.say("done"))
r2 = await ____(stack.runner, user_id="u-ana", session_id="s1", pending=pending)

crm = next(t["response"] for t in r2.tool_responses if t["name"] == "crm_lookup")
assert "user-delegated token" in crm["content"]
assert [(e.outcome, e.user) for e in stack.auth_manager.access_log] == [("uri_consent_required", "u-ana"), ("success", "u-ana")]
print(r2.summary())

**In one sentence:** "The agent never holds a standing wide token. It authenticates to
Auth Manager with its own identity and gets a per-call credential gated by IAM on the provider;
for user data that is a 3LO token naming both the user and the agent, and the access log shows both."